# ✂️ قصّاص — على السحابة مجانًا (بدون أي حساب جديد)

اضغط **Runtime → Run all** (أو Ctrl+F9) — أول مرة بتاخد حوالي 3-4 دقايق.

- لو ظهر تحذير إن النوتبوك "يشغّل كود من GitHub" → اضغط **Run anyway** (ده ريبو المشروع بتاعك)
- في آخر خلية هيظهرلك **لينك عام** — افتحه من أي جهاز (موبايل أو كمبيوتر) وابعت الفيديو عادي
- اللينك شغال طول ما الجلسة دي شغالة — نزّل نتيجتك قبل ما تقفل كولاب (التخزين مؤقت داخل الجلسة)
- الحد الأقصى للجلسة ~12 ساعة، وبعدين شغّل النوتبوك تاني وهتاخد لينك جديد

In [ ]:
# 1) تحميل الكود + تثبيت الحزم + البناء (أول مرة بتاخد حوالي 3 دقايق)
import os, subprocess, sys

def run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-3000:]); print(r.stderr[-3000:])
        sys.exit(f"❌ فشلت الخطوة: {cmd}")

print("⏳ [1/2] تحميل الكود + تثبيت الحزم...")
subprocess.run("apt-get -qq update && apt-get -qq install -y ffmpeg ca-certificates",
               shell=True, check=True)
if not os.path.exists(os.path.expanduser("~/.bun/bin/bun")):
    run("curl -fsSL https://bun.sh/install | bash")
os.environ["PATH"] = os.path.expanduser("~/.bun/bin") + ":" + os.environ["PATH"]
if not os.path.exists("/content/qattaas"):
    run("git clone --depth 1 https://github.com/mrmoneimofficial-del/autocut.git /content/qattaas")

print("⏳ [2/2] بناء التطبيق (أول مرة فقط)...")
run("cd /content/qattaas && bun install && bun run build")
print("✅ البناء خلص! شغّل الخلية اللي بعدها.")

In [ ]:
# 2) تشغيل السيرفر + فتح لينك عام (شغّل الخلية دي بعد ما الأولى تخلص)
import os, re, time, queue, threading, subprocess, urllib.request

print("⏳ تشغيل السيرفر...")
subprocess.run("pkill -f qattaas-deploy 2>/dev/null; pkill cloudflared 2>/dev/null; true",
               shell=True); time.sleep(2)

# نفس ترتيب ملفات الـ Dockerfile الرسمي
subprocess.run(
    "mkdir -p /content/qattaas-deploy && "
    "cp -r /content/qattaas/.next/standalone/. /content/qattaas-deploy/ && "
    "cp -r /content/qattaas/scripts /content/qattaas-deploy/",
    shell=True, check=True)

bun = os.path.expanduser("~/.bun/bin/bun")
subprocess.Popen(f"cd /content/qattaas-deploy && PORT=7860 HOSTNAME=0.0.0.0 {bun} server.js > server.log 2>&1",
                 shell=True)

ok = False
for _ in range(90):
    try:
        urllib.request.urlopen("http://localhost:7860", timeout=2); ok = True; break
    except Exception:
        time.sleep(1)
if not ok:
    print(open("/content/qattaas-deploy/server.log").read()[-3000:])
    raise RuntimeError("❌ السيرفر مش راضي يشتغل — شغّل الخلية الأولى تاني وبعدين دي")

if not os.path.exists("/usr/local/bin/cloudflared"):
    subprocess.run(
        "wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/"
        "cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared",
        shell=True, check=True)

print("⏳ فتح النفق العام...")
tunnel = subprocess.Popen(["/usr/local/bin/cloudflared", "tunnel", "--url", "http://localhost:7860",
                           "--no-autoupdate"],
                          stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, text=True)
lines = queue.Queue()
def _pump():
    for l in tunnel.stderr:
        lines.put(l)
threading.Thread(target=_pump, daemon=True).start()

url, deadline = None, time.time() + 90
while time.time() < deadline and not url:
    try:
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", lines.get(timeout=1))
        if m: url = m.group(0)
    except queue.Empty:
        pass
if not url:
    raise RuntimeError("❌ النفق فشل — شغّل الخلية دي تاني")

print()
print("=" * 62)
print("✅ التطبيق شغال! افتح اللينك ده من أي جهاز:")
print()
print("🚀 " + url)
print()
print("=" * 62)
print("⚠️ اللينك شغال طول ما الجلسة دي شغالة — سيب الصفحة مفتوحة")
print("💾 نزّل نتيجتك قبل ما تقفل كولاب — التخزين مؤقت داخل الجلسة")